In [ ]:
import os
import json
import uuid
from datetime import datetime
from typing import List, Dict, Any

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# ------------------------------
# 1. Experiment Manager for Prompt Versioning & Tracing
# ------------------------------
class ExperimentLogger:
    def __init__(self, log_dir="experiments"):
        os.makedirs(log_dir, exist_ok=True)
        self.log_dir = log_dir

    def log_experiment(self, experiment: Dict[str, Any]):
        exp_id = f"{experiment['model']}_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:6]}"
        filepath = os.path.join(self.log_dir, f"{exp_id}.json")
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(experiment, f, indent=2)
        return filepath


# ------------------------------
# 2. JSON Loader (Hierarchical Ingestion)
# ------------------------------
def load_hierarchical_json(json_dir: str) -> List[Document]:
    documents = []
    for root, _, files in os.walk(json_dir):
        for file in files:
            if file.endswith(".json"):
                with open(os.path.join(root, file), "r", encoding="utf-8") as f:
                    data = json.load(f)
                    ticker = os.path.basename(root)
                    documents.extend(flatten_hierarchy(data, ticker))
    return documents


def flatten_hierarchy(data: Dict[str, Any], ticker: str, section_path: str = "") -> List[Document]:
    docs = []
    section_title = data.get("section", "")
    new_path = f"{section_path}/{section_title}" if section_title else section_path

    # Paragraphs
    for para in data.get("paragraphs", []):
        docs.append(Document(page_content=para, metadata={
            "ticker": ticker,
            "section": new_path,
            "type": "paragraph"
        }))

    # Tables
    for table in data.get("tables", []):
        docs.append(Document(page_content=json.dumps(table), metadata={
            "ticker": ticker,
            "section": new_path,
            "type": "table"
        }))

    # Subsections
    for sub in data.get("subsections", []):
        docs.extend(flatten_hierarchy(sub, ticker, new_path))

    return docs


# ------------------------------
# 3. Build Vector Store (Chroma)
# ------------------------------
def build_vectorstore(docs: List[Document], persist_dir="chroma_db"):
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectordb = Chroma.from_documents(docs, embeddings, persist_directory=persist_dir)
    return vectordb


# ------------------------------
# 4. RAG Pipelines (Different Strategies)
# ------------------------------
class RAGStrategies:
    def __init__(self, vectordb, model="gpt-4o-mini"):
        self.llm = ChatOpenAI(model=model, temperature=0)
        self.retriever = vectordb.as_retriever(search_kwargs={"k": 5})

    def naive_rag(self, query: str):
        context = self.retriever.get_relevant_documents(query)
        prompt = PromptTemplate.from_template(
            "Answer the following question using context:\n\nContext: {context}\n\nQuestion: {question}"
        )
        return (prompt | self.llm).invoke({"context": self._docs_to_text(context), "question": query})

    def multi_query_rag(self, query: str):
        reformulations = [
            f"Alternative phrasing of: {query}",
            f"What does the filing say about {query}?",
            f"Find disclosures regarding {query}"
        ]
        docs = []
        for q in reformulations:
            docs.extend(self.retriever.get_relevant_documents(q))
        unique_docs = {d.page_content: d for d in docs}.values()
        return self._answer_with_context(query, list(unique_docs))

    def rag_fusion(self, query: str):
        docs = self.retriever.get_relevant_documents(query)
        summary_prompt = PromptTemplate.from_template(
            "Summarize the following retrieved contexts into a unified answer:\n{context}"
        )
        return (summary_prompt | self.llm).invoke({"context": self._docs_to_text(docs)})

    def decomposition_rag(self, query: str):
        sub_q_prompt = PromptTemplate.from_template(
            "Decompose the query into sub-questions:\n{query}"
        )
        sub_qs = (sub_q_prompt | self.llm).invoke({"query": query}).content.split("\n")
        docs = []
        for sq in sub_qs:
            docs.extend(self.retriever.get_relevant_documents(sq))
        return self._answer_with_context(query, docs)

    def step_back_rag(self, query: str):
        generalized = (PromptTemplate.from_template(
            "Reframe the query into a broader form:\n{query}"
        ) | self.llm).invoke({"query": query}).content
        docs = self.retriever.get_relevant_documents(generalized)
        return self._answer_with_context(query, docs)

    def hyde_rag(self, query: str):
        synthetic = (PromptTemplate.from_template(
            "Generate a hypothetical relevant document for query:\n{query}"
        ) | self.llm).invoke({"query": query}).content
        docs = self.retriever.get_relevant_documents(synthetic)
        return self._answer_with_context(query, docs)

    def routing_rag(self, query: str):
        router_prompt = PromptTemplate.from_template(
            "Choose the best RAG strategy (naive, fusion, step-back, hyde) for this query:\n{query}"
        )
        choice = (router_prompt | self.llm).invoke({"query": query}).content.strip().lower()
        strategy = getattr(self, f"{choice}_rag", self.naive_rag)
        return strategy(query)

    # --- Helper ---
    def _docs_to_text(self, docs: List[Document]) -> str:
        return "\n\n".join([f"[{d.metadata}] {d.page_content}" for d in docs])

    def _answer_with_context(self, query: str, docs: List[Document]):
        prompt = PromptTemplate.from_template(
            "Context:\n{context}\n\nQuestion: {question}\nAnswer concisely."
        )
        return (prompt | self.llm).invoke({"context": self._docs_to_text(docs), "question": query})


# ------------------------------
# 5. Runner Example
# ------------------------------
if __name__ == "__main__":
    json_dir = "json_data"   # <-- where hierarchical parsed JSON is stored
    documents = load_hierarchical_json(json_dir)
    print(f"Loaded {len(documents)} hierarchical chunks")

    vectordb = build_vectorstore(documents)
    rag = RAGStrategies(vectordb)

    query = "What were the net interest expenses for 2023 and 2024?"
    response = rag.multi_query_rag(query)

    # Log experiment
    logger = ExperimentLogger()
    log_file = logger.log_experiment({
        "model": "gpt-4o-mini",
        "strategy": "multi_query_rag",
        "query": query,
        "response": response.content,
        "config": {"retriever_k": 5}
    })
    print(f"Response: {response.content}")
    print(f"Experiment logged: {log_file}")
